# Comparativa: Modelos Clásicos vs Redes Neuronales
## Guía de Examen — Plantilla Reutilizable para Regresión y Clasificación

Este notebook cubre el patrón más probable del examen:
> *"Entrena un modelo con algoritmos clásicos y otro con una red neuronal. Compara los resultados."*

## Índice
1. **Por qué dos modelos** — qué aporta cada enfoque
2. **Regresión** — California Housing (o King County)
   - 2a. Preprocesamiento compartido
   - 2b. Modelo clásico: RandomForest / XGBoost
   - 2c. Red neuronal con PyTorch
   - 2d. Comparativa
3. **Clasificación** — Thyroid
   - 3a. Adaptaciones para clasificación
   - 3b. Red neuronal clasificadora
   - 3c. Comparativa con F2

## Regla de oro
> El modelo clásico (RandomForest, XGBoost) casi siempre gana en datos tabulares.
> La red neuronal es interesante para demostrar que la conoces, no porque vaya a ser mejor.


## 1. Por qué dos modelos

### Modelos clásicos (RandomForest, XGBoost)
| Ventaja | Detalle |
|---------|---------|
| Sin escalar el target | Los árboles usan umbrales, no gradientes |
| Toleran NaN nativamente (XGBoost) | No necesitan imputación explícita |
| Hiperparámetros intuitivos | `n_estimators`, `max_depth` son fáciles de entender |
| Resultados robustos | Poca varianza entre ejecuciones |
| Interpretables | `feature_importances_` de serie |

### Redes Neuronales (PyTorch)
| Ventaja | Cuándo importa |
|---------|---------------|
| Capturan relaciones no lineales complejas | Datos de imagen, texto, audio |
| Escalan con datos masivos | Millones de muestras |
| Transferencia de aprendizaje | Fine-tuning de modelos preentrenados |

### En datos tabulares (la mayoría del examen)
```
Gradient Boosting ≈ RandomForest >> Red Neuronal simple

La red neuronal puede igualar con mucho más ajuste de hiperparámetros,
pero para un examen de tiempo limitado no compensa.
```

**Conclusión**: entrena los dos, reporta métricas de ambos, y di por qué el clásico suele ganar.


---
## 2. Regresión — Plantilla Completa
### Adaptable a California Housing y King County

Este código es **autocontenido**: carga California Housing de sklearn si no hay CSV local.
Para el examen, solo cambias `DATA_PATH` y `TARGET_COL`.


### 2a. Preprocesamiento compartido

**Por qué compartir el preprocesamiento entre ambos modelos?**
Necesitamos comparar modelo clásico vs red neuronal en igualdad de condiciones.
Si usáramos features distintas, no sabríamos si la diferencia de rendimiento
viene del algoritmo o de las features.

El pipeline se **fit solo sobre train** y se **transforma** en val y test.
Esto evita leakage en ambos modelos.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ── Cargar datos ────────────────────────────────────────────────────────────
# Adaptación para California: cambia DATA_PATH y TARGET_COL
DATA_PATH  = Path("../proyects/COMPLETOS/california-e2e/data/housing.csv")
TARGET_COL = "median_house_value"

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    # Fallback: sklearn siempre tiene California Housing
    from sklearn.datasets import fetch_california_housing
    housing = fetch_california_housing(as_frame=True)
    df = housing.frame
    df = df.rename(columns={"MedHouseVal": TARGET_COL})
    df[TARGET_COL] = df[TARGET_COL] * 100_000  # escalar a dólares
    print("Usando California Housing de sklearn (sin CSV local)")

print(f"Dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Target: {TARGET_COL}  |  Media: ${df[TARGET_COL].mean():,.0f}")


In [ ]:
# ── Split ────────────────────────────────────────────────────────────────────
# Para California: split estratificado por income
# Para King County: sustituir por split temporal (iloc[:split_idx])

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].values.astype(float)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=42
)
# 3 conjuntos: train (entreno) / val (early stopping + elijo modelo) / test (evaluación final)

print(f"Train: {len(X_train):,}  |  Val: {len(X_val):,}  |  Test: {len(X_test):,}")


In [ ]:
# ── Pipeline de preprocesamiento ─────────────────────────────────────────────
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), cat_cols),
], remainder="drop")

# FIT solo sobre train → transforma val y test con los mismos parámetros
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)

n_features = X_train_proc.shape[1]
print(f"Features tras preprocesamiento: {n_features}")


### 2b. Modelo Clásico: RandomForest + XGBoost

**Por qué RandomForest como baseline?**
Es robusto, no necesita escalar el target, interpreta bien los datos tabulares
y tiene pocos hiperparámetros críticos. Es la referencia honesta contra la que comparar la red neuronal.

**Por qué también XGBoost?**
En regresión tabular, XGBoost generalmente supera a RandomForest.
Si tenemos ambos, elegimos el mejor en val para comparar con la red neuronal.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluar_regresion(nombre, y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"{nombre}:  RMSE={rmse:>10,.0f}$   MAE={mae:>10,.0f}$   R²={r2:.4f}")
    return {"modelo": nombre, "rmse": rmse, "mae": mae, "r2": r2}

# ── RandomForest ─────────────────────────────────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_proc, y_train)
resultados_val = []
resultados_val.append(evaluar_regresion("RandomForest (val)", y_val, rf.predict(X_val_proc)))


In [ ]:
# ── XGBoost ──────────────────────────────────────────────────────────────────
# XGBoost maneja NaN nativamente, pero aquí ya los imputamos en el pipeline
try:
    from xgboost import XGBRegressor
    xgb = XGBRegressor(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )
    xgb.fit(X_train_proc, y_train,
            eval_set=[(X_val_proc, y_val)],
            verbose=False)
    resultados_val.append(evaluar_regresion("XGBoost (val)", y_val, xgb.predict(X_val_proc)))
except ImportError:
    print("XGBoost no instalado — usando solo RandomForest")


### 2c. Red Neuronal con PyTorch

Aquí están las decisiones clave que hay que saber explicar.

#### Por qué escalar el target (`y`) en regresión?
Los modelos de árboles hacen splits en umbrales → la escala del target no importa.
Las redes neuronales calculan gradientes → si `y` está en el rango [50000, 500000],
el MSE es del orden de `10^10`, los gradientes son enormes y el modelo diverge.

**Solución**: escalar `y` a media≈0, std≈1 para entrenar, desescalar para evaluar.
Usamos `log1p` primero porque los precios de casas tienen distribución sesgada:
```
y_scaled = StandardScaler().fit_transform(log1p(y_train))
y_real = expm1(scaler.inverse_transform(y_scaled_pred))
```

#### Por qué DataLoader y no pasar X directamente?
PyTorch no entrena con arrays completos. Usa **mini-batches**:
- En cada paso del optimizador, calcula el gradiente en un subconjunto de datos
- Más eficiente en memoria: no necesita cargar todo el dataset en GPU
- Más rápido: el gradiente se actualiza N veces por epoch (N = n_muestras / batch_size)

#### Por qué `model.train()` y `model.eval()`?
- `BatchNorm` calcula estadísticas distintas en train y en evaluación
- `Dropout` desactiva neuronas aleatoriamente en train, no en evaluación
Si olvidas poner `model.eval()` antes de evaluar, el dropout activa ruido y las métricas son peores de lo real.

#### Por qué Early Stopping?
Sin early stopping, la red neuronal sigue entrenando hasta el epoch máximo.
Si el val loss empieza a subir (overfitting), queremos volver al mejor modelo, no al último.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# ── Dispositivo ──────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando: {device}")

# ── Escalar target ───────────────────────────────────────────────────────────
# Por qué log1p? Los precios tienen distribución sesgada → log los normaliza
# Por qué solo fit sobre y_train? Evitar leakage del val/test
y_scaler = StandardScaler()
y_train_s = y_scaler.fit_transform(np.log1p(y_train).reshape(-1, 1))  # fit aquí
y_val_s   = y_scaler.transform(np.log1p(y_val).reshape(-1, 1))        # solo transform
y_test_s  = y_scaler.transform(np.log1p(y_test).reshape(-1, 1))

print(f"y_train original: media={y_train.mean():,.0f}  std={y_train.std():,.0f}")
print(f"y_train_scaled:   media={y_train_s.mean():.3f}  std={y_train_s.std():.3f}")


In [ ]:
# ── DataLoaders ──────────────────────────────────────────────────────────────
# Por qué float32? PyTorch trabaja en float32 por defecto (float64 es más lento en GPU)
def make_loader(X_np, y_np, batch_size=256, shuffle=False):
    X_t = torch.tensor(X_np, dtype=torch.float32)
    y_t = torch.tensor(y_np, dtype=torch.float32)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

BATCH_SIZE = 256
train_loader = make_loader(X_train_proc, y_train_s, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_val_proc,   y_val_s,   BATCH_SIZE)
test_loader  = make_loader(X_test_proc,  y_test_s,  BATCH_SIZE)

print(f"Batches por epoch: {len(train_loader)}")
print(f"Muestras por batch: {BATCH_SIZE}")


In [ ]:
# ── Arquitectura ─────────────────────────────────────────────────────────────
class RegresionNet(nn.Module):
    """
    Red neuronal para regresión tabular.

    Decisiones de diseño:
    - Capas decrecientes: 256→128→64 (embudo)
    - BatchNorm antes de ReLU: estabiliza el entrenamiento
    - Dropout: regularización para evitar overfitting
    - Sin activación en la salida: queremos valores sin límites (precio puede ser cualquier número)
    """
    def __init__(self, n_input: int, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            # Capa 1: n_input → 256
            nn.Linear(n_input, 256),
            nn.BatchNorm1d(256),   # normaliza activaciones dentro del batch
            nn.ReLU(),             # activa no-linealidad
            nn.Dropout(dropout),   # apaga 30% de neuronas aleatoriamente

            # Capa 2: 256 → 128
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),  # menos dropout cerca de la salida

            # Capa 3: 128 → 64
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            # Salida: 64 → 1 (sin activación = regresión)
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x)

model_reg = RegresionNet(n_features).to(device)
n_params = sum(p.numel() for p in model_reg.parameters())
print(f"Parámetros totales: {n_params:,}")


In [ ]:
# ── Bucle de entrenamiento con Early Stopping ────────────────────────────────
def train_regression(
    model, train_loader, val_loader,
    epochs=200, lr=1e-3, patience=15
):
    """
    Entrena el modelo y aplica early stopping.

    Por qué MSELoss sobre el target escalado?
    Si usáramos MSE sobre dólares, el loss sería ~10^10 → gradientes enormes → inestabilidad.
    Con el target escalado (std≈1), el loss es del orden de 0.1-1.0 → estable.

    Por qué Adam en vez de SGD?
    Adam ajusta la tasa de aprendizaje por parámetro automáticamente.
    Converge más rápido y es menos sensible al learning rate inicial.
    """
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss   = float("inf")
    best_weights    = None
    best_epoch      = 0
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        # ── FASE TRAIN ─────────────────────────────────────────────
        model.train()          # activa Dropout y BatchNorm en modo train
        running = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()          # limpiar gradientes del paso anterior
            preds = model(X_batch)         # forward pass
            loss  = criterion(preds, y_batch)  # calcular MSE
            loss.backward()                # backpropagation
            optimizer.step()               # actualizar pesos
            running += loss.item() * len(X_batch)

        train_loss = running / len(train_loader.dataset)
        history["train_loss"].append(train_loss)

        # ── FASE VALIDACIÓN ────────────────────────────────────────
        model.eval()           # desactiva Dropout, BatchNorm usa stats globales
        running_val = 0.0
        with torch.no_grad():  # no calcular gradientes (ahorra memoria)
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                preds   = model(X_batch)
                loss    = criterion(preds, y_batch)
                running_val += loss.item() * len(X_batch)

        val_loss = running_val / len(val_loader.dataset)
        history["val_loss"].append(val_loss)

        # ── EARLY STOPPING ─────────────────────────────────────────
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch    = epoch
            best_weights  = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping en epoch {epoch+1}. Mejor: epoch {best_epoch+1}")
                break

        if (epoch + 1) % 25 == 0:
            print(f"  Epoch {epoch+1:3d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

    model.load_state_dict(best_weights)   # restaurar el mejor modelo
    return history, best_epoch

print("Entrenando red neuronal...")
history_reg, best_ep = train_regression(model_reg, train_loader, val_loader)


In [ ]:
# ── Predicción y desescalado ──────────────────────────────────────────────────
@torch.no_grad()
def predecir(model, loader, y_scaler):
    """
    Genera predicciones y las devuelve en las unidades originales (dólares).

    Por qué expm1(inverse_transform(...))?
    Aplicamos las transformaciones en orden inverso:
      1. inverse_transform: deshace el StandardScaler (vuelve al espacio log)
      2. expm1: deshace el log1p (vuelve a dólares)
    """
    model.eval()
    preds_scaled = []
    for X_batch, _ in loader:
        X_batch = X_batch.to(device)
        preds_scaled.append(model(X_batch).cpu().numpy())
    preds_scaled = np.concatenate(preds_scaled).reshape(-1, 1)
    return np.expm1(y_scaler.inverse_transform(preds_scaled).squeeze())

nn_val_pred  = predecir(model_reg, val_loader, y_scaler)
nn_test_pred = predecir(model_reg, test_loader, y_scaler)

resultados_val.append(evaluar_regresion("NeuralNet (val)", y_val, nn_val_pred))


### 2d. Comparativa y elección del modelo final

**Proceso correcto de elección**:
1. Ajustar hiperparámetros usando solo train + val
2. Comparar todos los modelos en **val** → elegir el mejor
3. Evaluar el modelo elegido en **test una sola vez**

Si comparamos en test para elegir, el test ya no es una estimación honesta del rendimiento real.


In [ ]:
import matplotlib.pyplot as plt

# ── Tabla comparativa en val ──────────────────────────────────────────────────
print("=" * 65)
print("COMPARATIVA EN VALIDATION SET")
print("=" * 65)
for r in resultados_val:
    print(f"  {r['modelo']:<25} RMSE={r['rmse']:>10,.0f}$  R²={r['r2']:.4f}")

mejor = min(resultados_val, key=lambda r: r["rmse"])
print(f"\n→ Mejor modelo en val: {mejor['modelo']} (RMSE={mejor['rmse']:,.0f}$)")


In [ ]:
# ── Curvas de aprendizaje de la red neuronal ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_reg["train_loss"], label="Train loss (MSE escalado)", alpha=0.8)
ax.plot(history_reg["val_loss"],   label="Val loss (MSE escalado)",   alpha=0.8)
ax.axvline(best_ep, color="red", linestyle="--",
           label=f"Mejor epoch ({best_ep+1}) — Early Stopping aquí")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss (target escalado)")
ax.set_title("Curvas de aprendizaje — Red Neuronal")
ax.legend()

# Qué buscar:
# - Train y val bajando juntos → aprendiendo bien
# - Val empieza a subir → overfitting → Early Stopping actúa
# - Train > val en algunas epochs → normal con Dropout (añade ruido en train)
plt.tight_layout()
plt.show()


In [ ]:
# ── Evaluación final en test ──────────────────────────────────────────────────
print("=" * 65)
print("EVALUACIÓN FINAL EN TEST SET")
print("=" * 65)
print("(Solo se reporta — la elección ya se hizo sobre val)\n")

evaluar_regresion("RandomForest (test)", y_test, rf.predict(X_test_proc))
evaluar_regresion("NeuralNet    (test)", y_test, nn_test_pred)

# ── Gráfica: predicción vs real ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (nombre, y_pred) in zip(axes, [
    ("RandomForest", rf.predict(X_test_proc)),
    ("Red Neuronal",  nn_test_pred),
]):
    ax.scatter(y_test, y_pred, alpha=0.3, s=8)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, "r--", lw=2, label="Predicción perfecta")
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    ax.set_title(f"{nombre}\nRMSE = ${rmse:,.0f}")
    ax.set_xlabel("Precio real ($)")
    ax.set_ylabel("Precio predicho ($)")

plt.tight_layout()
plt.show()


---
## 3. Clasificación — Thyroid (y cualquier problema de clases)

### Diferencias clave respecto a regresión

| Aspecto | Regresión | Clasificación |
|---------|-----------|---------------|
| Target | Escalar continuo → escalar con StandardScaler | Etiqueta categórica → LabelEncoder (int) |
| Loss | MSELoss | CrossEntropyLoss |
| Capa de salida | `Linear(64, 1)` sin activación | `Linear(64, n_clases)` sin activación |
| Predicción | `model(X)` directo | `model(X).argmax(dim=1)` |
| Métrica | RMSE, MAE, R² | F1, F2, accuracy |
| Target scaling | Sí (escalar y desescalar) | No (las etiquetas son enteros) |

### Por qué CrossEntropyLoss en vez de MSELoss?
CrossEntropyLoss = Softmax + NegativeLogLikelihood.
Convierte las activaciones de la última capa en probabilidades por clase y penaliza
más los errores con alta confianza equivocada que los errores con baja confianza.
MSE en clasificación no tiene este comportamiento.


In [ ]:
# ── Cargar datos Thyroid ─────────────────────────────────────────────────────
THYROID_PATH = Path("datasets/thyroid/thyroidDF.csv")
if not THYROID_PATH.exists():
    THYROID_PATH = Path("../proyects-upgrades/datasets/thyroid/thyroidDF.csv")

if not THYROID_PATH.exists():
    print("AVISO: No se encontró thyroidDF.csv.")
    print("Generando datos sintéticos para demostrar la arquitectura...")
    from sklearn.datasets import make_classification
    X_syn, y_syn = make_classification(
        n_samples=3000, n_features=20, n_informative=10,
        n_classes=3, random_state=42
    )
    df_thy = pd.DataFrame(X_syn, columns=[f"f{i}" for i in range(20)])
    df_thy["target"] = ["negative", "hypothyroid", "hyperthyroid"][y_syn] if False else [
        {0: "negative", 1: "hypothyroid", 2: "hyperthyroid"}[v] for v in y_syn
    ]
else:
    df_thy = pd.read_csv(THYROID_PATH)

    def simplify(v):
        if any(c in str(v) for c in ["A","B","C","D"]): return "hyperthyroid"
        if any(c in str(v) for c in ["E","F","G","H"]): return "hypothyroid"
        return "negative"

    df_thy["target"] = df_thy["target"].apply(simplify)
    df_thy = df_thy.dropna(subset=["target"])
    df_thy = df_thy.drop(columns=["patient_id", "referral_source"], errors="ignore")

print(df_thy["target"].value_counts())


In [ ]:
from sklearn.preprocessing import LabelEncoder

X_thy = df_thy.drop(columns=["target"]).copy()
# Edades imposibles → NaN
if "age" in X_thy.columns:
    X_thy.loc[X_thy["age"] > 150, "age"] = np.nan

le = LabelEncoder()
y_thy = le.fit_transform(df_thy["target"])   # "hyperthyroid"→0, "hypothyroid"→1, "negative"→2
n_clases = len(le.classes_)
print(f"Clases: {le.classes_} → {list(range(n_clases))}")

# Split estratificado
X_thy_tv, X_thy_test, y_thy_tv, y_thy_test = train_test_split(
    X_thy, y_thy, test_size=0.2, random_state=42, stratify=y_thy
)
X_thy_tr, X_thy_val, y_thy_tr, y_thy_val = train_test_split(
    X_thy_tv, y_thy_tv, test_size=0.2, random_state=42, stratify=y_thy_tv
)

# Preprocesamiento
num_thy = X_thy_tr.select_dtypes(include="number").columns.tolist()
cat_thy = X_thy_tr.select_dtypes(exclude="number").columns.tolist()

pre_thy = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
    ]), num_thy),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), cat_thy),
])

X_thy_tr_p  = pre_thy.fit_transform(X_thy_tr)
X_thy_val_p = pre_thy.transform(X_thy_val)
X_thy_tst_p = pre_thy.transform(X_thy_test)
n_feat_thy  = X_thy_tr_p.shape[1]
print(f"Features: {n_feat_thy}")


### 3b. Modelo Clásico para clasificación

Igual que en regresión, primero el baseline.
Recordatorio: usar `class_weight="balanced"` para datos desbalanceados.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, fbeta_score

def thyroid_f2(y_true, y_pred, le):
    labels_enfermos = [i for i, c in enumerate(le.classes_) if c != "negative"]
    return fbeta_score(
        y_true, y_pred,
        beta=2,
        labels=labels_enfermos,
        average="macro",
        zero_division=0,
    )

rf_cls = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf_cls.fit(X_thy_tr_p, y_thy_tr)
rf_val_pred = rf_cls.predict(X_thy_val_p)

print("RandomForest — Validation:")
print(classification_report(y_thy_val, rf_val_pred,
                            target_names=le.classes_, zero_division=0))
print(f"F2 (clases enfermas): {thyroid_f2(y_thy_val, rf_val_pred, le):.4f}")


### 3c. Red Neuronal para Clasificación

**Diferencias respecto a la red de regresión**:
1. La capa de salida tiene `n_clases` neuronas, no 1
2. La función de pérdida es `CrossEntropyLoss` (incorpora Softmax internamente)
3. Para predecir la clase: `logits.argmax(dim=1)` — la neurona con mayor activación
4. **No escalamos `y`** — las etiquetas son enteros y CrossEntropyLoss las espera así
5. Para datos desbalanceados: pasar `weight` a `CrossEntropyLoss`

**Qué es CrossEntropyLoss?**
```
CrossEntropy = -log(P(clase_correcta))
```
Si el modelo predice 90% de probabilidad a la clase correcta → loss pequeño
Si el modelo predice 10% de probabilidad a la clase correcta → loss grande


In [ ]:
class ClasificadorNet(nn.Module):
    """
    Red neuronal para clasificación multiclase.
    La ÚNICA diferencia vs RegresionNet:
    - La capa de salida tiene n_clases neuronas
    - No hay activación en la salida (CrossEntropyLoss la aplica internamente)
    """
    def __init__(self, n_input: int, n_clases: int, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_input, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, n_clases),   # ← n_clases neuronas, no 1
        )

    def forward(self, x):
        return self.net(x)             # devuelve logits (sin softmax)

model_cls = ClasificadorNet(n_feat_thy, n_clases).to(device)
print(f"Parámetros: {sum(p.numel() for p in model_cls.parameters()):,}")


In [ ]:
# ── Pesos de clase para CrossEntropyLoss ─────────────────────────────────────
# Equivalente a class_weight="balanced" de sklearn
# Peso de clase c = n_total / (n_clases × n_muestras_de_clase_c)
class_counts  = np.bincount(y_thy_tr)
class_weights = len(y_thy_tr) / (n_clases * class_counts)
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Pesos por clase:")
for cls, w in zip(le.classes_, class_weights):
    print(f"  {cls}: {w:.3f}")

# CrossEntropyLoss con pesos: penaliza más los errores en clases raras
criterion_cls = nn.CrossEntropyLoss(weight=weights_tensor)


In [ ]:
# ── DataLoaders para clasificación ───────────────────────────────────────────
# Las etiquetas son enteros (long), no float
def make_loader_cls(X_np, y_np, batch_size=128, shuffle=False):
    X_t = torch.tensor(X_np, dtype=torch.float32)
    y_t = torch.tensor(y_np, dtype=torch.long)   # ← long para CrossEntropyLoss
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

train_loader_cls = make_loader_cls(X_thy_tr_p,  y_thy_tr,  shuffle=True)
val_loader_cls   = make_loader_cls(X_thy_val_p, y_thy_val)
test_loader_cls  = make_loader_cls(X_thy_tst_p, y_thy_test)


In [ ]:
# ── Bucle de entrenamiento para clasificación ─────────────────────────────────
optimizer_cls = torch.optim.Adam(model_cls.parameters(), lr=1e-3)

best_val_f2   = -1.0
best_weights_cls = None
best_ep_cls   = 0
patience_cls  = 20
no_improve    = 0

for epoch in range(150):
    # ── Train ──────────────────────────────────────────────────────────────
    model_cls.train()
    for X_b, y_b in train_loader_cls:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer_cls.zero_grad()
        logits = model_cls(X_b)          # shape: (batch, n_clases)
        loss   = criterion_cls(logits, y_b)  # CrossEntropy espera (logits, labels_long)
        loss.backward()
        optimizer_cls.step()

    # ── Val F2 ─────────────────────────────────────────────────────────────
    model_cls.eval()
    all_preds = []
    with torch.no_grad():
        for X_b, _ in val_loader_cls:
            logits = model_cls(X_b.to(device))
            preds  = logits.argmax(dim=1)   # clase con mayor probabilidad
            all_preds.extend(preds.cpu().numpy())

    val_f2 = thyroid_f2(y_thy_val, all_preds, le)

    # Early stopping por F2 (maximizar, no minimizar)
    if val_f2 > best_val_f2:
        best_val_f2  = val_f2
        best_ep_cls  = epoch
        best_weights_cls = {k: v.clone() for k, v in model_cls.state_dict().items()}
        no_improve   = 0
    else:
        no_improve += 1
        if no_improve >= patience_cls:
            print(f"Early stopping en epoch {epoch+1}. Mejor: {best_ep_cls+1} (F2={best_val_f2:.4f})")
            break

    if (epoch + 1) % 30 == 0:
        print(f"  Epoch {epoch+1:3d}  val_F2={val_f2:.4f}")

model_cls.load_state_dict(best_weights_cls)


In [ ]:
# ── Comparativa final ─────────────────────────────────────────────────────────
model_cls.eval()
nn_preds_test = []
with torch.no_grad():
    for X_b, _ in test_loader_cls:
        logits = model_cls(X_b.to(device))
        nn_preds_test.extend(logits.argmax(dim=1).cpu().numpy())

rf_preds_test = rf_cls.predict(X_thy_tst_p)

print("=" * 60)
print("COMPARATIVA FINAL EN TEST SET")
print("=" * 60)

for nombre, preds in [("RandomForest", rf_preds_test), ("NeuralNet", nn_preds_test)]:
    f2 = thyroid_f2(y_thy_test, preds, le)
    print(f"\n{nombre} — F2={f2:.4f}:")
    print(classification_report(y_thy_test, preds,
                                target_names=le.classes_, zero_division=0))


## Resumen — Lo que hay que saber decir en el examen

### Sobre la red neuronal

> *"La arquitectura es una red densa con capas 256→128→64→salida.
> Uso BatchNorm para estabilizar el entrenamiento y Dropout como regularizador.
> Para regresión, escalo el target con log1p + StandardScaler porque las redes son
> sensibles a la escala de la variable objetivo y un MSE de 10^10 destabiliza los gradientes.
> Entreno con Adam y aplico early stopping monitorizando el val loss.
> Para clasificación, uso CrossEntropyLoss con pesos de clase inversamente proporcionales
> a la frecuencia, equivalente al class_weight='balanced' de sklearn."*

### Por qué el modelo clásico suele ganar en datos tabulares

> *"Los árboles de decisión (y sus ensambles) son invariantes a la escala,
> toleran NaN, no necesitan normalización del target y capturan interacciones
> entre variables sin necesidad de diseñar la arquitectura.
> Para que una red neuronal iguale a XGBoost en datos tabulares se necesita
> mucho más ajuste de hiperparámetros de arquitectura.
> Sin embargo, conocer PyTorch es importante para problemas de imagen, texto y audio
> donde los modelos clásicos no escalan."*

### Checklist antes de entregar

| Paso | Regresión | Clasificación |
|------|-----------|---------------|
| Escalar target | ✅ log1p + StandardScaler | ❌ No (enteros) |
| Desescalar predicciones | ✅ expm1 + inverse_transform | ❌ No |
| Loss | MSELoss | CrossEntropyLoss |
| Pesos de clase | class_weight="balanced" en RF | weight= en CrossEntropyLoss |
| Salida de la red | Linear(64, 1) | Linear(64, n_clases) |
| Predicción | model(X) directo | model(X).argmax(dim=1) |
| model.train() en loop | ✅ | ✅ |
| model.eval() en val/test | ✅ | ✅ |
| torch.no_grad() al evaluar | ✅ | ✅ |
| Early stopping | val loss (minimizar) | val F2 (maximizar) |
